# Gemma 4 E2B × TinyCeNN Memory Fusion — Replace ALL text attention

This copies the working sequential/live-progress workflow used by the Qwen3.5 experiment, but uses **`google/gemma-4-E2B`**.

The test targets **all 35 text decoder attention layers** (both sliding and full attention). A layer is selected immediately if it passes the strict NMSE/cosine/ΔNLL gates. If no strict candidate is found after the configured rounds, the run selects the **closest measured candidate** and labels it `closest_fallback` instead of pretending it passed.

Gemma 4 E2B is large (~10 GB BF16 checkpoint), so the trainer loads **one text model** and switches dual attention wrappers between the frozen original path and Memory Fusion path. This avoids keeping two full base-model copies on the GPU. The multimodal vision/audio encoders are not replaced in this experiment.


In [1]:
import os,sys,json,subprocess,tempfile,shutil
from pathlib import Path
assert subprocess.run(['nvidia-smi'],check=False).returncode==0,'Enable a GPU runtime'
REPO=Path(tempfile.mkdtemp(prefix='gemma4-e2b-memoryfusion-'))/'TinyCeNN-LM'
subprocess.run(['git','clone','--quiet','https://github.com/vtavakkoli/TinyCeNN-LM.git',str(REPO)],check=True)
subprocess.run(['git','fetch','origin','main'],cwd=REPO,check=True)
subprocess.run(['git','reset','--hard','origin/main'],cwd=REPO,check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.17.0','datasets','huggingface_hub','accelerate','safetensors','pytest','pandas','matplotlib'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'--no-deps'],check=True)
os.environ['PYTHONPATH']=os.pathsep.join([str(REPO),str(REPO/'src')]); sys.path[:0]=[str(REPO),str(REPO/'src')]
import torch
SOURCE=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('Source:',SOURCE); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


Source: 81a1c69fe79facfdd00bd3e4760aa675b6b18528
GPU: NVIDIA L4


In [2]:
from huggingface_hub import HfApi,login,notebook_login
from google.colab import userdata
try: token=userdata.get('HF_TOKEN')
except Exception: token=None
if token: os.environ['HF_TOKEN']=token; login(token=token,add_to_git_credential=False)
else: notebook_login()
print('✅ HF user:',HfApi().whoami().get('name'))


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ HF user: vtava


In [3]:
from google.colab import drive
from huggingface_hub import HfApi
from transformers import AutoConfig
drive.mount('/content/drive')
BASE_MODEL='google/gemma-4-E2B'
MODEL_REVISION=HfApi().model_info(BASE_MODEL).sha
PROFILE='balanced' # @param ['smoke','balanced','extended']
RESET_PROGRESS=False # @param {type:'boolean'}
PUBLISH_TO_HF=True # @param {type:'boolean'}
PROFILES={
 'smoke':dict(context=32,probe=32,steps=50,check=25,force_rounds=1),
 'balanced':dict(context=64,probe=64,steps=150,check=25,force_rounds=2),
 'extended':dict(context=128,probe=128,steps=250,check=25,force_rounds=3),
}
RUN=PROFILES[PROFILE]; FEATURE_DIM=32; MEMORY_RANK=64; SEED=73
CONTEXT=RUN['context']; PROBE_CONTEXT=RUN['probe']; MAX_LAYER_STEPS=RUN['steps']; CHECK_EVERY=RUN['check']; FORCE_AFTER_ROUNDS=RUN['force_rounds']
HF_MODEL_REPO='vtava/Gemma4-E2B-MemoryFusion-AllAttention'; HF_PRIVATE=False
OUTPUT_DIR=Path('/content/drive/MyDrive/TinyCeNN-LM/gemma4-e2b-memory-fusion-all-r64')
if RESET_PROGRESS and OUTPUT_DIR.exists(): shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
cfg=AutoConfig.from_pretrained(BASE_MODEL,revision=MODEL_REVISION).get_text_config(decoder=True)
TARGET_LAYERS=list(range(cfg.num_hidden_layers)); LAYER_TYPES=list(cfg.layer_types)
FULL_LAYERS=[i for i,t in enumerate(LAYER_TYPES) if t=='full_attention']; SLIDING_LAYERS=[i for i,t in enumerate(LAYER_TYPES) if t=='sliding_attention']
assert cfg.model_type=='gemma4_text',cfg.model_type
assert len(TARGET_LAYERS)==35,TARGET_LAYERS
print(json.dumps({'profile':PROFILE,'model':BASE_MODEL,'revision':MODEL_REVISION,'targets':len(TARGET_LAYERS),'full':FULL_LAYERS,'sliding_count':len(SLIDING_LAYERS),'context':CONTEXT,'probe_context':PROBE_CONTEXT,'max_steps_per_round':MAX_LAYER_STEPS,'force_after_rounds':FORCE_AFTER_ROUNDS,'output':str(OUTPUT_DIR),'hf_repo':HF_MODEL_REPO},indent=2))


Mounted at /content/drive


config.json:   0%|          | 0.00/4.91k [00:00<?, ?B/s]

{
  "profile": "balanced",
  "model": "google/gemma-4-E2B",
  "revision": "d29ff6b45f081a49ee2733a859c9c9c2d95d1a6f",
  "targets": 35,
  "full": [
    4,
    9,
    14,
    19,
    24,
    29,
    34
  ],
  "sliding_count": 28,
  "context": 64,
  "probe_context": 64,
  "max_steps_per_round": 150,
  "force_after_rounds": 2,
  "output": "/content/drive/MyDrive/TinyCeNN-LM/gemma4-e2b-memory-fusion-all-r64",
  "hf_repo": "vtava/Gemma4-E2B-MemoryFusion-AllAttention"
}


In [4]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES='',OMP_NUM_THREADS='1',MKL_NUM_THREADS='1'); env['TINYCENN_PARENT_BACKUP_ACTIVE']='1'
r=subprocess.run([sys.executable,'-m','pytest','-q','tests/test_gemma4_memory_fusion.py'],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f'preflight failed: {r.returncode}')
print('✅ Gemma 4 Memory Fusion preflight passed')


....                                                                     [100%]
4 passed in 19.20s

✅ Gemma 4 Memory Fusion preflight passed


## Saved progress before training

Green `✓` = strict gate pass. Blue `≈` = closest fallback after the configured training rounds. The saved dashboard survives Colab disconnects.


In [5]:
from tinycenn_lm.gemma4_memory_fusion_colab import show_progress,run_live_training,DEFAULT_COMPLETION_PROMPTS
PROGRESS_BEFORE=show_progress(OUTPUT_DIR,TARGET_LAYERS)


## Train / resume all 35 text-attention layers

Every line is streamed immediately and saved to Drive. Strict acceptance is preferred; when strict quality is not reached after `FORCE_AFTER_ROUNDS`, the best measured candidate by normalized gate distance is selected as `closest_fallback` and the conversion continues.


In [6]:
cmd=[sys.executable,'-u',str(REPO/'scripts'/'train_gemma4_memory_fusion_all_attention.py'),'--base-model',BASE_MODEL,'--model-revision',MODEL_REVISION,'--output-dir',str(OUTPUT_DIR),'--feature-dim',str(FEATURE_DIM),'--memory-rank',str(MEMORY_RANK),'--context-length',str(CONTEXT),'--probe-context',str(PROBE_CONTEXT),'--seed',str(SEED),'--min-layer-steps','50','--max-layer-steps',str(MAX_LAYER_STEPS),'--check-every',str(CHECK_EVERY),'--layer-lr','0.0002','--teacher-alpha-start','0.9','--teacher-alpha-end','0.0','--accept-nmse','0.2','--accept-cosine','0.9','--accept-incremental-delta-nll','0.015','--accept-cumulative-delta-nll','0.05','--force-all','--force-after-rounds',str(FORCE_AFTER_ROUNDS),'--max-runtime-minutes','420','--resume']
run_env=dict(os.environ)
return_code=run_live_training(cmd=cmd,repo=REPO,output_dir=OUTPUT_DIR,target_layers=TARGET_LAYERS,max_step=MAX_LAYER_STEPS,env=run_env)
if return_code: raise RuntimeError(f'trainer failed with exit code {return_code}; see {OUTPUT_DIR/"last_colab_run.log"}')
print('✅ trainer returned normally')
PROGRESS_AFTER=show_progress(OUTPUT_DIR,TARGET_LAYERS)


▶ TRAIN / RESUME — Gemma 4 E2B all 35 text-attention layers
Command: /usr/bin/python3 -u /tmp/gemma4-e2b-memoryfusion-ftoq8q4p/TinyCeNN-LM/scripts/train_gemma4_memory_fusion_all_attention.py --base-model google/gemma-4-E2B --model-revision d29ff6b45f081a49ee2733a859c9c9c2d95d1a6f --output-dir /content/drive/MyDrive/TinyCeNN-LM/gemma4-e2b-memory-fusion-all-r64 --feature-dim 32 --memory-rank 64 --context-length 64 --probe-context 64 --seed 73 --min-layer-steps 50 --max-layer-steps 150 --check-every 25 --layer-lr 0.0002 --teacher-alpha-start 0.9 --teacher-alpha-end 0.0 --accept-nmse 0.2 --accept-cosine 0.9 --accept-incremental-delta-nll 0.015 --accept-cumulative-delta-nll 0.05 --force-all --force-after-rounds 2 --max-runtime-minutes 420 --resume
Live log: /content/drive/MyDrive/TinyCeNN-LM/gemma4-e2b-memory-fusion-all-r64/last_colab_run.log
[TinyCeNN][PROCESS START] train_gemma4_memory_fusion_all_attention
[TinyCeNN][BACKUP REQUIRED][DIRECT] https://huggingface.co/vtava/TinyCeNN-LM-Colab-

RuntimeError: trainer failed with exit code 74; see /content/drive/MyDrive/TinyCeNN-LM/gemma4-e2b-memory-fusion-all-r64/last_colab_run.log

In [ ]:
for name in ('colab_runtime_status.json','sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json'):
    p=OUTPUT_DIR/name
    if p.exists(): print('\n###',name); print(p.read_text(encoding='utf-8'))


## Sample comparison — original Gemma 4 vs current Memory Fusion model

This uses one loaded Gemma 4 text model and switches the wrapped attention layers between `original` and `memory`. The current unselected/in-progress layer is excluded; only layers saved in `sequential_progress.pt` are used.


In [ ]:
import torch,gc
from transformers import AutoTokenizer,Gemma4ForCausalLM
from tinycenn_lm.gemma4_memory_fusion import Gemma4MemoryFusionConfig,ensure_dual_layers,load_selected_memory_state,set_attention_mode,structural_summary
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); DTYPE=torch.bfloat16 if DEVICE.type=='cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE.type=='cuda' else torch.float32)
pp=OUTPUT_DIR/'sequential_progress.pt'; payload=torch.load(pp,map_location='cpu',weights_only=False) if pp.exists() else None
replaced=[int(x) for x in payload.get('replaced_layers',[])] if payload else []; reports=list(payload.get('layer_reports',[])) if payload else []
mf_cfg=Gemma4MemoryFusionConfig.from_dict(payload['config']) if payload else Gemma4MemoryFusionConfig(feature_dim=FEATURE_DIM,memory_rank=MEMORY_RANK)
tok=AutoTokenizer.from_pretrained(BASE_MODEL,revision=MODEL_REVISION)
model=Gemma4ForCausalLM.from_pretrained(BASE_MODEL,revision=MODEL_REVISION,dtype=DTYPE,attn_implementation='sdpa',low_cpu_mem_usage=True).to(DEVICE).eval()
model.config.use_cache=False
if replaced: ensure_dual_layers(model,mf_cfg,replaced); load_selected_memory_state(model,payload['memory_state'],replaced)
print('Replaced layers:',len(replaced),'/',len(TARGET_LAYERS)); print(json.dumps(structural_summary(model),indent=2))
@torch.no_grad()
def complete(mode,prompt):
    if replaced: set_attention_mode(model,mode)
    batch=tok(prompt,return_tensors='pt').to(DEVICE)
    out=model.generate(**batch,max_new_tokens=80,do_sample=False,use_cache=False,pad_token_id=tok.eos_token_id)
    return tok.decode(out[0,batch['input_ids'].shape[1]:],skip_special_tokens=True).strip()
examples=[]
for i,prompt in enumerate(DEFAULT_COMPLETION_PROMPTS,1):
    old=complete('original',prompt); new=complete('memory',prompt)
    print('\n'+'='*100); print(f'PROMPT {i}:',prompt); print('\nORIGINAL GEMMA 4:\n',old); print('\nMEMORY FUSION:\n',new)
    examples.append({'prompt':prompt,'original':old,'memory_fusion':new,'replaced_layers':replaced})
(OUTPUT_DIR/'completion_examples.json').write_text(json.dumps(examples,indent=2,ensure_ascii=False),encoding='utf-8')
print('✅ saved',OUTPUT_DIR/'completion_examples.json')


## Publish adapter + results to Hugging Face

Only TinyCeNN replacement weights and experiment reports are uploaded; the 10 GB Google base weights are not duplicated. Fallback layers remain explicitly labeled in the model card.


In [ ]:
from tinycenn_lm.gemma4_memory_fusion import save_adapter
PACKAGE=Path('/content/gemma4-e2b-memory-fusion-hf')
if PACKAGE.exists(): shutil.rmtree(PACKAGE)
PACKAGE.mkdir(parents=True); (PACKAGE/'training').mkdir()
if replaced: save_adapter(model,PACKAGE,config=mf_cfg,base_model=BASE_MODEL,replaced_layers=replaced,layer_reports=reports,metadata={'base_revision':MODEL_REVISION,'source_repo':'vtavakkoli/TinyCeNN-LM','source_commit':SOURCE})
for name in ('sequential_run_status.json','sequential_progress.json','sequential_in_progress.json','sequential_training_report.json','completion_examples.json','last_colab_run.log','colab_runtime_status.json'):
    src=OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src,PACKAGE/name)
for name in ('sequential_in_progress.pt','sequential_progress.pt','gemma4_memory_fusion_full_state.pt'):
    src=OUTPUT_DIR/name
    if src.exists(): shutil.copy2(src,PACKAGE/'training'/name)
strict=sorted({int(r['layer']) for r in reports if r.get('selected') and r.get('selection')=='strict'}); fallback=sorted({int(r['layer']) for r in reports if r.get('selected') and r.get('selection')=='closest_fallback'})
card=f'''---
library_name: transformers
base_model: {BASE_MODEL}
license: apache-2.0
tags: [gemma4, tinycenn, memory-fusion, recurrent-attention, research]
---
# Gemma 4 E2B × TinyCeNN Memory Fusion — All Text Attention

Experimental text-decoder attention conversion for `{BASE_MODEL}`. The target is all 35 text self-attention layers; Gemma 4 vision/audio encoders are not replaced or evaluated here.

**Base revision:** `{MODEL_REVISION}`
**Source commit:** `{SOURCE}`
**Replaced:** `{len(replaced)}/35`
**Strict gate passes:** `{strict}`
**Closest fallback layers:** `{fallback}`

Strict gates: NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ 0.015, cumulative ΔNLL ≤ 0.05. A `closest_fallback` layer did **not** pass all strict gates; after the configured training rounds, the checkpoint with the smallest normalized gate-miss distance was selected so the all-attention conversion could continue. See the included JSON reports and `completion_examples.json` for the measured results.
'''
(PACKAGE/'README.md').write_text(card,encoding='utf-8')
api=HfApi()
if PUBLISH_TO_HF:
    api.create_repo(HF_MODEL_REPO,repo_type='model',private=HF_PRIVATE,exist_ok=True); api.upload_folder(repo_id=HF_MODEL_REPO,repo_type='model',folder_path=str(PACKAGE),commit_message=f'Update Gemma 4 E2B Memory Fusion replaced={len(replaced)}/35'); print('✅ Published: https://huggingface.co/'+HF_MODEL_REPO)
else: print('Publication disabled; package ready at',PACKAGE)
